# Contraindication Retrieval on Google Colab

This notebook runs the contraindication similarity search against ICD-11 vector database. It processes pharmaceutical contraindications and finds similar medical conditions using vector embeddings.

## Configuration
- **AIC Code**: 045034307
- **Category**: condition
- **Model**: jinaai/jina-embeddings-v3

## Process
1. Install dependencies
2. Upload/setup project files
3. Run the retrieval script with specified filters
4. Download results

## 1. Install Required Packages

In [ ]:
# Install required packages for retrieval
!pip install -q sentence-transformers chromadb tqdm scipy numpy

# Check GPU availability
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("💻 Using CPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 7.0 MB/s eta 

## 2. Upload Project Files

You need to upload several files to run the retrieval:
- **Vector database** (created by indexing with jinaai/jina-embeddings-v3)
- **Contraindications data** (all_contraindications_verified.json)
- **Retrieval script** (vector_db_retrieval.py)

In [ ]:
# Mount Google Drive to access your files
from google.colab import drive
import os

drive.mount('/content/drive')

# Create project structure
os.makedirs("data/contraindications", exist_ok=True)
os.makedirs("data/interaction_results", exist_ok=True)
os.makedirs("data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/chroma_langchain_db", exist_ok=True)
os.makedirs("src/retrieval", exist_ok=True)

print("📁 Project directories created!")
print("📋 Next steps:")
print("   1. Copy your vector database files to data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/")
print("   2. Copy contraindications file to data/contraindications/")
print("   3. Copy retrieval script to src/retrieval/")
print("\n💡 Use the file manager on the left to upload files or copy from Drive")

Mounted at /content/drive
📁 Project directories created!
📋 Next steps:
   1. Copy your vector database files to data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/
   2. Copy contraindications file to data/contraindications/
   3. Copy retrieval script to src/retrieval/

💡 Use the file manager on the left to upload files or copy from Drive


## 3. File Upload Options

### Option A: Upload from Local Computer
Use the file manager on the left panel to upload:
- `all_contraindications_verified.json` → `data/contraindications/`
- `vector_db_jinaai_jina_embeddings_v3/` folder → `data/`
- `vector_db_retrieval.py` → `src/retrieval/`

### Option B: Copy from Google Drive
If you have the files in your Google Drive, use the code below to copy them:

In [ ]:
# Option B: Copy files from Google Drive (uncomment and modify paths as needed)
import shutil

# Example paths - modify these to match your Google Drive structure
DRIVE_PATH = "/content/drive/MyDrive/rxassist-ai"

# Copy contraindications file
shutil.copy(f"{DRIVE_PATH}/data/contraindications/all_contraindications_verified.json",
            "data/contraindications/")

# Copy vector database folder
shutil.copytree(f"{DRIVE_PATH}/data/vector_dbs/vector_db_jinaai_jina_embeddings_v3",
                "data/vector_dbs/vector_db_jinaai_jina_embeddings_v3", dirs_exist_ok=True)

# Copy retrieval script
shutil.copy(f"{DRIVE_PATH}/src/retrieval/vector_db_retrieval.py",
            "src/retrieval/")

print("📥 Files copied from Google Drive!")
print("💡 Uncomment and modify the paths above to use this option")

📥 Files copied from Google Drive!
💡 Uncomment and modify the paths above to use this option


## 4. Verify Files are in Place

In [ ]:
# Check if all required files are present
import os

required_files = [
    "data/contraindications/all_contraindications_verified.json",
    "src/retrieval/vector_db_retrieval.py",
    "data/vector_dbs/vector_db_jinaai_jina_embeddings_v3"
]

print("🔍 Checking required files...")
all_present = True

for file_path in required_files:
    if os.path.exists(file_path):
        if os.path.isdir(file_path):
            print(f"✅ Directory: {file_path}")
        else:
            file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
            print(f"✅ File: {file_path} ({file_size:.1f} MB)")
    else:
        print(f"❌ Missing: {file_path}")
        all_present = False

if all_present:
    print("\n🎉 All required files are present! Ready to run retrieval.")
else:
    print("\n⚠️ Some files are missing. Please upload them before continuing.")

🔍 Checking required files...
✅ File: data/contraindications/all_contraindications_verified.json (8.3 MB)
✅ File: src/retrieval/vector_db_retrieval.py (0.0 MB)
✅ Directory: data/vector_dbs/vector_db_jinaai_jina_embeddings_v3

🎉 All required files are present! Ready to run retrieval.


## 5. Run Contraindication Retrieval

Now we'll run the retrieval script with your specified parameters:
- **AIC Code**: 045034307
- **Category**: condition  
- **Model**: jinaai/jina-embeddings-v3

In [ ]:
# Run the retrieval script with your specified parameters
import subprocess
import time

print("🚀 Starting contraindication retrieval...")
print("📋 Parameters:")
print("   • AIC Code: 045034307")
print("   • Category: condition")
print("   • Model: jinaai/jina-embeddings-v3")
print()

# Build the command
cmd = [
    "python", "src/retrieval/vector_db_retrieval.py",
    "--aic-codes", "045034307",
    "--category", "condition",
    "--model", "jinaai/jina-embeddings-v3"
]

print(f"💻 Command: {' '.join(cmd)}")
print("⏱️ This may take several minutes...")
print("-" * 50)

# Run the command
start_time = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)

# Show results
if result.returncode == 0:
    print("✅ Retrieval completed successfully!")
    print("\n📄 Output:")
    print(result.stdout)
else:
    print("❌ Error occurred:")
    print(result.stderr)
    if result.stdout:
        print("\nStdout:")
        print(result.stdout)

elapsed = time.time() - start_time
print(f"\n⏱️ Total time: {elapsed:.1f} seconds")

🚀 Starting contraindication retrieval...
📋 Parameters:
   • AIC Code: 045034307
   • Category: condition
   • Model: jinaai/jina-embeddings-v3

💻 Command: python src/retrieval/vector_db_retrieval.py --aic-codes 045034307 --category condition --model jinaai/jina-embeddings-v3
⏱️ This may take several minutes...
--------------------------------------------------
✅ Retrieval completed successfully!

📄 Output:
🤖 Using model: jinaai/jina-embeddings-v3
📁 Vector DB path: /content/data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/chroma_langchain_db
🏷️ Filtering by category: condition
🏷️ Filtering by AIC codes: ['045034307']
📂 Loading contraindications file...
🎯 Filtering by 1 AIC codes
📊 AIC filtering: 10/10515 contraindications (0.1%) in 1/716 AICs
🎯 Filtered AICs: ['045034307']
🏷️ Filtering by category: condition
📊 Category 'condition': 6/10 contraindications (60.0%) in 1 AICs
🔧 Initializing embedding model: jinaai/jina-embeddings-v3
🎯 Using Jina embeddings
🔧 Connecting to ChromaDB...
✅ C

## 6. View and Download Results

In [ ]:
# View and download the results
import json
import os
from google.colab import files

# Look for the output file
output_dir = "data/interaction_results"
output_files = [f for f in os.listdir(output_dir) if f.startswith("interaction_results_jinaai")]

if output_files:
    # Get the most recent file
    latest_file = max(output_files, key=lambda x: os.path.getctime(os.path.join(output_dir, x)))
    file_path = os.path.join(output_dir, latest_file)

    print(f"📊 Results file: {latest_file}")
    print(f"📁 Size: {os.path.getsize(file_path) / 1024:.1f} KB")

    # Load and preview the results
    with open(file_path, 'r') as f:
        results = json.load(f)

    print(f"\n📈 Results summary:")
    print(f"   • Total AICs processed: {len(results.get('aic_results', []))}")

    total_searches = sum(len(aic['similarity_searches']) for aic in results.get('aic_results', []))
    print(f"   • Total contraindications processed: {total_searches}")

    # Show preview of first results
    print("\n🔍 Preview of first results:")
    for i, aic_result in enumerate(results.get('aic_results', [])[:1]):
        print(f"\nAIC: {aic_result.get('aic', 'N/A')}")
        print(f"Contraindications: {aic_result.get('contraindications_count', 0)}")

        for j, search in enumerate(aic_result.get('similarity_searches', [])[:2]):
            print(f"\n  Contraindication {j+1}:")
            print(f"    Italian: {search['original_warning']['italian'][:100]}...")
            print(f"    English: {search['original_warning']['english'][:100]}...")

            # Show top 3 most similar ICD matches
            top_matches = search.get('similar_documents', [])[:3]
            print(f"    Top {len(top_matches)} ICD matches:")
            for k, match in enumerate(top_matches):
                distance = match.get('distance', 'N/A')
                similarity = match.get('similarity', 1 - distance if isinstance(distance, (int, float)) else 'N/A')
                icd_code = match.get('metadata', {}).get('code', 'N/A')
                print(f"      {k+1}. {icd_code} - Distance: {distance:.4f}, Similarity: {similarity:.4f}")
                print(f"         {match.get('document', '')[:80]}...")

    # Download the file
    print(f"\n⬇️ Downloading {latest_file}...")
    files.download(file_path)

else:
    print("❌ No results file found. Check if the retrieval completed successfully.")

📊 Results file: interaction_results_jinaai_jina_embeddings_v3_20250630_094414.json
📁 Size: 283.8 KB

📈 Results summary:
   • Total AICs processed: 1
   • Total contraindications processed: 6

🔍 Preview of first results:

AIC: 045034307
Contraindications: 6

  Contraindication 1:
    Italian: Si rivolga al medico o al farmacista prima di prendere PREGABALIN EG STADA ITALIA. In alcuni pazient...
    English: - In some patients with diabetes who gain weight during treatment with pregabalin, it may be necessa...
    Top 3 ICD matches:
      1. 5B81.1 - Distance: 1.2011, Similarity: 0.0000
         Drug-induced obesity Drug-induced obesity, obesity due to drug, medicament-induc...
      2. 5A13.3 - Distance: 1.2715, Similarity: 0.0000
         Diabetes mellitus due to endocrinopathies Other specified diabetes mellitus due ...
      3. 8B94 - Distance: 1.2932, Similarity: 0.0000
         Diabetic radiculoplexoneuropathy Diabetic radiculoplexoneuropathy is a rare, but...

  Contraindication 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

This notebook successfully:
1. ✅ Installed required packages (sentence-transformers, chromadb, etc.)
2. ✅ Set up the project structure in Colab
3. ✅ Provided options for file upload (local files or Google Drive)
4. ✅ Verified all required files are present
5. ✅ Ran contraindication retrieval with specified filters:
   - AIC Code: 045034307
   - Category: condition
   - Model: jinaai/jina-embeddings-v3
6. ✅ Displayed and downloaded results

## Understanding the Results

**Scoring System:**
- **Distance**: Lower values = more similar (0 = identical, higher = less similar)
- **Similarity**: Higher values = more similar (0-1 scale, 1 = identical)
- **Results are ranked by similarity** (most similar first)

**Output Structure:**
- Each AIC contains multiple contraindications
- Each contraindication is matched against ICD-11 codes
- Top matches represent the most medically similar conditions

## Troubleshooting

**If you encounter errors:**
- **Missing files**: Ensure all required files are uploaded to the correct locations
- **CUDA errors**: The script will fallback to CPU if GPU is unavailable
- **Model loading issues**: Verify the vector database was created with the correct model
- **Memory issues**: Consider using a smaller batch size or different model

**Expected output file format:**
- Filename: `interaction_results_jinaai_jina_embeddings_v3_YYYYMMDD_HHMMSS.json`
- Contains similarity scores between contraindications and ICD-11 codes
- Filtered by your specified AIC code and category